# BrailleVision — YOLOv8 Training on Google Colab
**Run all cells top to bottom. Takes ~60-90 min on a free T4 GPU.**

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')
!nvidia-smi | head -10

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────
!pip install -q ultralytics opencv-python-headless numpy pyyaml
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Upload your BrailleVision project ────────────────────
# Zip your entire BrailleVision folder on your PC and upload it here.
# On Windows: right-click BrailleVision folder > Send to > Compressed (zipped) folder

from google.colab import files
import zipfile, os, pathlib

print('Select your BrailleVision.zip file to upload...')
uploaded = files.upload()  # <-- A file picker will appear

# Extract zip
for filename in uploaded.keys():
    print(f'Extracting {filename}...')
    with zipfile.ZipFile(filename, 'r') as z:
        z.extractall('/content/')
    print('Extracted.')

In [ ]:
# ── Cell 4: Find project root (handles nested zip extraction) ─────
import os
from pathlib import Path

# Search for the project root — the folder containing 'training/' and 'dataset/'
PROJECT_ROOT = None
for root, dirs, files_list in os.walk('/content/'):
    if 'training' in dirs and ('dataset' in dirs or 'frontend' in dirs):
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Could not find BrailleVision project root! Did the zip extract correctly?')

print(f'Project root: {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)
!ls

In [ ]:
# ── Cell 5: Download real Braille datasets ─────────────────────────
!python training/scripts/download_datasets.py --skip-synthetic
print('\nReal datasets done.')

In [ ]:
# ── Cell 6: Generate 800 synthetic Braille images ──────────────────
!python training/scripts/generate_synthetic.py --count 800
print('\nSynthetic generation done.')

In [ ]:
# ── Cell 7: Merge & split into train/val/test ──────────────────────
!python training/scripts/merge_and_split.py
!python training/scripts/verify_dataset.py
print('\nDataset ready.')
!cat dataset/data.yaml

In [ ]:
# ── Cell 8: TRAIN ─────────────────────────────────────────────────
# ~60-90 min on T4 GPU
!python training/scripts/train.py \
    --epochs 100 \
    --batch 16 \
    --device 0 \
    --name braille_dot_v1

In [ ]:
# ── Cell 9: Evaluate ──────────────────────────────────────────────
!python training/scripts/evaluate.py --skip-e2e
!cat training/runs/eval_report.md

In [ ]:
# ── Cell 10: Download weights ──────────────────────────────────────
from google.colab import files
import os

model_pt   = 'model/best.pt'
model_onnx = 'model/model.onnx'

if os.path.exists(model_pt):
    files.download(model_pt)
    print('Downloaded best.pt')
else:
    print('ERROR: best.pt not found. Check training output above.')

if os.path.exists(model_onnx):
    files.download(model_onnx)
    print('Downloaded model.onnx')

print('\nPlace both files in the model/ folder of your local project.')